In [2]:
import torch
import numpy as np
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
 
# prepare dataset
 
 
class DiabetesDataset(Dataset):
    def __init__(self, filepath):
        xy = np.loadtxt(filepath, delimiter=',', dtype=np.float32)
        self.len = xy.shape[0] # shape(多少行，多少列)
        self.x_data = torch.from_numpy(xy[:, :-1])
        self.y_data = torch.from_numpy(xy[:, [-1]])
 
    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]
 
    def __len__(self):
        return self.len
 
 
dataset = DiabetesDataset('data\diabetes.csv')
train_loader = DataLoader(dataset=dataset, batch_size=32, shuffle=True, num_workers=0) #num_workers 多线程
 
 
# design model using class
 
 
class Model(torch.nn.Module):
    def __init__(self):
        super(Model, self).__init__()
        self.linear1 = torch.nn.Linear(8, 6)
        self.linear2 = torch.nn.Linear(6, 4)
        self.linear3 = torch.nn.Linear(4, 1)
        self.sigmoid = torch.nn.Sigmoid()
 
    def forward(self, x):
        x = self.sigmoid(self.linear1(x))
        x = self.sigmoid(self.linear2(x))
        x = self.sigmoid(self.linear3(x))
        return x
 
 
model = Model()
 
# construct loss and optimizer
criterion = torch.nn.BCELoss(reduction='mean')
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
 
# training cycle forward, backward, update
if __name__ == '__main__':
    for epoch in range(100):
        for i, data in enumerate(train_loader, 0): # train_loader 是先shuffle后mini_batch
            inputs, labels = data
            y_pred = model(inputs)
            loss = criterion(y_pred, labels)
            print(epoch, i, loss.item())
 
            optimizer.zero_grad()
            loss.backward()
 
            optimizer.step()

0 0 0.8575294613838196
0 1 0.7802136540412903
0 2 0.7642709016799927
0 3 0.7929903864860535
0 4 0.8202593326568604
0 5 0.7040743827819824
0 6 0.8033642768859863
0 7 0.7877193689346313
0 8 0.8414466381072998
0 9 0.7977795004844666
0 10 0.7690500617027283
0 11 0.7685293555259705
0 12 0.715362012386322
0 13 0.740604043006897
0 14 0.8051692843437195
0 15 0.7902168035507202
0 16 0.7886154651641846
0 17 0.7250903844833374
0 18 0.7122100591659546
0 19 0.8349912166595459
0 20 0.7953879237174988
0 21 0.8052440285682678
0 22 0.7564931511878967
0 23 0.7817492485046387
1 0 0.6870735883712769
1 1 0.8107080459594727
1 2 0.7640098929405212
1 3 0.76271653175354
1 4 0.7830041646957397
1 5 0.7604862451553345
1 6 0.7694375514984131
1 7 0.7782685160636902
1 8 0.7159973978996277
1 9 0.8159655332565308
1 10 0.7443592548370361
1 11 0.724179744720459
1 12 0.7334564924240112
1 13 0.7423118352890015
1 14 0.70390385389328
1 15 0.7222638130187988
1 16 0.7407596111297607
1 17 0.7944048643112183
1 18 0.826397657394

## Train / Test Split

The cell above trains on the **whole** dataset, so there's no way to tell if the model generalizes. Below we split the data into a **training set** (used to learn) and a **test set** (held out, used only to measure). `random_split` divides the dataset into non-overlapping subsets, and each gets its own `DataLoader`.

In [3]:
from torch.utils.data import random_split

# reuse the DiabetesDataset defined above
dataset = DiabetesDataset('data\\diabetes.csv')

# 80% train, 20% test
torch.manual_seed(42)                    # makes the split reproducible
n_total = len(dataset)
n_train = int(0.8 * n_total)
n_test = n_total - n_train
train_set, test_set = random_split(dataset, [n_train, n_test])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=0)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=0)

print('total:', n_total, '| train:', n_train, '| test:', n_test)

total: 759 | train: 607 | test: 152


### Train on the training set

A fresh model, trained only on `train_loader`. We track the average loss per epoch.

In [4]:
# a fresh model (reuses the Model class defined above)
model = Model()
criterion = torch.nn.BCELoss(reduction='mean')
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

for epoch in range(100):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        y_pred = model(inputs)
        loss = criterion(y_pred, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    if epoch % 10 == 0:
        print(f'epoch {epoch:3d}  train loss {running_loss / len(train_loader):.4f}')

epoch   0  train loss 0.7427
epoch  10  train loss 0.6440
epoch  20  train loss 0.6442
epoch  30  train loss 0.6439
epoch  40  train loss 0.6436
epoch  50  train loss 0.6439
epoch  60  train loss 0.6433
epoch  70  train loss 0.6437
epoch  80  train loss 0.6430
epoch  90  train loss 0.6428


### Test on the held-out set

Evaluate on `test_loader` — data the model never saw during training.

- `model.eval()` switches to evaluation mode.
- `torch.no_grad()` turns off gradient tracking (faster, less memory — we're not learning).
- The model already outputs a probability (its last layer is sigmoid), so we threshold at 0.5 to get a 0/1 prediction and compare to the true label.

In [5]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        probs = model(inputs)                 # already a probability (sigmoid output)
        preds = (probs >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f'Test accuracy: {correct / total:.4f}  ({correct}/{total})')

Test accuracy: 0.6447  (98/152)
